In [2]:
import os
import numpy as np

midi_file_paths = []
for root, dirs, files in os.walk("POP909-Dataset/POP909"):
    for file in files:
        if file.endswith(".mid") and not root.endswith("versions"):
            midi_file_paths.append(os.path.join(root, file))

print(f"\nTotal MIDI files found: {len(midi_file_paths)}")


Total MIDI files found: 909


In [3]:
import mido

def ExtractMelodyTones(file_path):

    abs_path = file_path

    midi_mel = mido.MidiFile(abs_path)
    seq = []

    for msg in midi_mel.tracks[1]: #Vi vil kun have melodistemen
        if msg.type == 'note_on' and msg.velocity > 0:  # Kun note_on with velocity > 0
            if seq == []:
                seq.append(msg.note)
            elif msg.note != seq[-1]:
                seq.append(msg.note)

    return seq

def ConvertToFreqs(seq):
    return [round(440 * (2**((tone - 69)/12)), 3) for tone in seq]

melodies_midi = [ExtractMelodyTones(path) for path in midi_file_paths]
melodies_freqs = [ConvertToFreqs(seq) for seq in melodies_midi]

In [4]:
print("There is " + str(len(melodies_freqs)) + " sequences")
print("with an average length of " + str(sum([len(seq) for seq in melodies_freqs]) / len(melodies_freqs)) + " tones")

individual_tones = set()
for seq in melodies_midi:
    for tone in seq:
        individual_tones.add(tone)

print("Consisting of " + str(len(individual_tones)) + " unique tones")

There is 909 sequences
with an average length of 260.1771177117712 tones
Consisting of 55 unique tones


### Variable Order Markov Model

First we need the n-grams the maximum order of seven (septogram)

In [5]:
n_grams = []
max_order = 7

for i in range(max_order+1):
    i_gram = []
    if i == 0:
        for seq in melodies_midi:
            for tone in seq:
                i_gram.append(tuple([tone]))
    else:
        for seq in melodies_midi:
            for j, tone in enumerate(seq[:-i]):
                i_gram.append(tuple(seq[j:j+i+1]))

    n_grams.append(i_gram)



In [6]:
for i, n_gram in enumerate(n_grams):
    print("There is " + str(len(n_gram)) + " " + str(i) + "-grams in the corpus")

There is 236501 0-grams in the corpus
There is 235592 1-grams in the corpus
There is 234683 2-grams in the corpus
There is 233774 3-grams in the corpus
There is 232865 4-grams in the corpus
There is 231956 5-grams in the corpus
There is 231047 6-grams in the corpus
There is 230138 7-grams in the corpus


In [7]:
from collections import defaultdict, Counter

n_gram_counts = [defaultdict(int) for _ in range(8)]

for order, i_grams in enumerate(n_grams):
    for gram in i_grams:
        n_gram_counts[order][gram] += 1 

Calculates the probabilities

$$
p_{esc}=number of distinct next symbols/total count for this context​
$$

In [180]:
import random

def PredictNext(seq):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = min(len(seq), 7)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob

    probs.pop(seq[-1], None)

    min_val1, max_val1 = 0.01, 1 

    new_keys = dict(filter(lambda x: x[1]<= max_val1 and x[1]>= min_val1, probs.items()))
    real = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real_prob = probs[real]
    probs.pop(real, None)

    return real, real_prob


Generating a simple melodi with pure probability

In [182]:
def ToneRandChoice(probs):
    total = sum(probs.values())
    choice = np.random.random() * total
    for key, value in probs.items():
        if choice < value:
            return key, value
        choice -= value


def GenerateMelody(probe, length):
    context = (probe,)
    all_probs = []
    for i in range(length):
        new_tone, prob = PredictNext(context)
        all_probs.append(prob)
        context += (new_tone,)
    return list(context), all_probs 


Using this to create binary trees for sequential melody production
$$
N=2^0+2^1+...+2^{l-1}=2^l-1
$$

In [ ]:
import math

def GenerateBinaryTree(probe, length):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        fst_pos = 2**layer - 1
        for node in range(0,2**layer, 2):
            parents = [math.ceil((node+fst_pos)/2)-1]
            for i in range(layer-2):
                parents.append(math.ceil(parents[-1]/2) - 1)
            context = tuple([tree[i] for i in parents])
            tone1, prob1 = PredictNext(context)
            tone2, prob2 = PredictNext(context)
            while tone2 == tone1:
                tone2, prob2 = PredictNext(context)
            choice = np.random.random()
            if choice > 0.5:
                tone1, tone2 = tone2, tone1
                prob1, prob2 = prob2, prob1
            tree[fst_pos + node] = tone1
            tree[fst_pos + node + 1] = tone2

            prob_tree[fst_pos + node] = prob1
            prob_tree[fst_pos + node + 1] = prob2
    return tree, prob_tree

[67, 74, 79, 76, 71, 72, 76, 71, 73, 79, 76, 84, 77, 83, 76, 73, 71, 71, 69, 71, 76, 73, 76, 75, 77, 77, 81, 81, 76, 76, 78]
[1, 0.032376790342134115, 0.010017930643978717, 0.21756326275615928, 0.11114698732579932, 0.04550550594908382, 0.1195976336605145, 0.1659647069536131, 0.06861925914034317, 0.03207103102528522, 0.3461234174287086, 0.025722269559301396, 0.3946269637098621, 0.03059243841782315, 0.3446159716699517, 0.08881639672866441, 0.3022182453277567, 0.16657892170253497, 0.25559329103277323, 0.11962035808114493, 0.6143410023552308, 0.043680669277667755, 0.37938583014147814, 0.10550998159986237, 0.5621709901026898, 0.6601687343969385, 0.1416784042558193, 0.4872040537674697, 0.39076932861165864, 0.3446159716699517, 0.11070457898315865]


Just for testing the tree, i will make a function to choose a random sequence

In [205]:
def TreeRandSeq(tree, length):
    seq = [tree[0]]
    parent = 0
    for i in range(length-1):
        choice = np.random.random()
        if choice > 0.5:
            seq.append(tree[2*(parent+1)-1])
            parent = 2*(parent+1)-1
        else:
            seq.append(tree[2*(parent+1)])
            parent = 2*(parent+1)
    return seq

SomeTree = ConvertToFreqs(GenerateBinaryTree(77,3))
print(SomeTree)
print(TreeRandSeq(SomeTree, 3))

[698.456, 830.609, 587.33, 739.989, 698.456, 698.456, 493.883]
[698.456, 830.609, 739.989]


Generating a melody with change
- Input is probe, position and suprise-factor

In [184]:
import random

def PredictNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = min(len(seq), 7)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob

    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    if surprisal1:
        min_val1, max_val1 = 0.01, 0.05 
    else:
        min_val1, max_val1 = 0.15, 1
    
    if surprisal2:
        min_val2, max_val2 = 0.01, 0.05 
    else:
        min_val2, max_val2 = 0.15, 1

    new_keys = dict(filter(lambda x: x[1]<= max_val1 and x[1]>= min_val1, probs.items()))
    real = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real_prob = probs[real]
    probs.pop(real, None)


    new_keys = dict(filter(lambda x: x[1]<= max_val2 and x[1]>= min_val2, probs.items()))
    alt = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    alt_prob = probs[alt]

    return (real, real_prob), (alt, alt_prob)


def GenerateMelodyWithChange(probe, length, pos, surprise):
    context = (probe,)
    probs = []
    for i in range(length-1):
        if i+1 == pos:
            tone, alt = PredictNextWithAlternative(context, surprise)
            if tone == False or alt == False:
                return False
            new_tone, val = tone
        else:
            new_tone, val = PredictNext(context)
        context += (new_tone,)
        probs.append(val)
    return list(context), probs, alt

mel = GenerateMelodyWithChange(61, 8, 3, (True, False))
if not mel == False:
    tones, probs, alts = mel
    print(tones)
    print(probs)
    print(alts)
else:
    print("Error")

[61, 68, 66, 71, 68, 66, 64, 61]
[0.05481092377644102, 0.6444634657631046, 0.04494261523902906, 0.48496540617951056, 0.8815471179663638, 0.8264375369293261, 0.9877876829608169]
(68, 0.4540468137205612)


Now the same for trees

In [186]:
def PredictTwoNextWithAlternative(seq, surprise):
    probs = defaultdict(float)
    remaining_prob = 1.0
    order = len(seq)

    for o in range(order, 0, -1):
        context = seq[order - o :]
        total = sum(
            n_gram_counts[o][context + (s,)] for s in individual_tones
        )
        if total == 0:
            continue
        distinct = sum(
            1 for s in individual_tones if n_gram_counts[o][context + (s,)] > 0
        )
        p_esc = distinct / (total + distinct)

        for s in individual_tones:
            count = n_gram_counts[o][context + (s,)]
            if count > 0:
                probs[s] += remaining_prob * (1 - p_esc) * (count / total)
        
        remaining_prob *= p_esc

    unseen_prob = remaining_prob / len(individual_tones)
    for s in individual_tones:
        probs[s] += unseen_prob
    
    probs.pop(seq[-1], None)

    surprisal1, surprisal2 = surprise

    if surprisal1:
        min_val1, max_val1 = 0.01, 0.05 
    else:
        min_val1, max_val1 = 0.15, 1
    
    if surprisal2:
        min_val2, max_val2 = 0.01, 0.05 
    else:
        min_val2, max_val2 = 0.15, 1


    new_keys = dict(filter(lambda x: x[1]<= max_val1 and x[1]>= min_val1, probs.items()))
    real1 = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real1_prob = probs[real1]
    probs.pop(real1, None)
    new_keys.pop(real1, None)

    real2 = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    real2_prob = probs[real1]
    probs.pop(real2, None)

    new_keys = dict(filter(lambda x: x[1]<= max_val2 and x[1]>= min_val2, probs.items()))
    alt = random.choices(list(new_keys.keys()), weights=list(new_keys.values()), k=1)[0]
    alt_prob = probs[alt]

    return (real1, real1_prob), (real2, real2_prob), (alt, alt_prob)

def GenerateBinaryTreeWithChange(probe, length, pos, surprise):
    tree = [0 for _ in range(2**length-1)]
    prob_tree = [0 for _ in range(2**length-1)]
    alts = []

    tree[0] = probe
    prob_tree[0] = 1
    for layer in range(1,length,1):
        if layer+1 == pos:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)
                context = tuple([tree[i] for i in parents])
                real1, real2, alt = PredictTwoNextWithAlternative(context, surprise)
                if real1 == False or real2 == False or alt == False:
                    return False
                tone1, prob1 = real1
                tone2, prob2 = real2
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2
                alts.append(alt)

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2
        else:
            fst_pos = 2**layer - 1
            for node in range(0,2**layer, 2):
                parents = [math.ceil((node+fst_pos)/2)-1]
                for i in range(layer-2):
                    parents.append(math.ceil(parents[-1]/2) - 1)
                context = tuple([tree[i] for i in parents])
                tone1, prob1 = PredictNext(context)
                tone2, prob2 = PredictNext(context)
                while tone2 == tone1:
                    tone2, prob2 = PredictNext(context)
                choice = np.random.random()
                if choice > 0.5:
                    tone1, tone2 = tone2, tone1
                    prob1, prob2 = prob2, prob1
                tree[fst_pos + node] = tone1
                tree[fst_pos + node + 1] = tone2

                prob_tree[fst_pos + node] = prob1
                prob_tree[fst_pos + node + 1] = prob2
    return tree, prob_tree, alts

mel = GenerateBinaryTreeWithChange(61, 5, 2, (True, False))
if not mel == False:
    tones, probs, alts = mel
    print(tones)
    print(probs)
    print(alts)
else:
    print("Error")

[61, 56, 60, 59, 61, 72, 65, 59, 66, 59, 58, 62, 67, 63, 67, 58, 59, 54, 59, 59, 58, 58, 54, 74, 70, 63, 62, 63, 58, 57, 65]
[1, 0.0172216103250586, 0.0, 0.1610294117647059, 0.06911764705882353, 0.01686103012633625, 0.06350826044703596, 0.43833904727535183, 0.09374323348971492, 0.08269686907020873, 0.5489761227071474, 0.21588313896987366, 0.17843780369290574, 0.28930723775608036, 0.053495607255366194, 0.04060357271743053, 0.43833904727535183, 0.013289760348583878, 0.6204946736704224, 0.32677620055466355, 0.2805329513939571, 0.6452683859467541, 0.2917414179748145, 0.0849459426627794, 0.04551141885325558, 0.017334031827016524, 0.3873041180758018, 0.406161603830327, 0.18459130077475985, 0.05577194977277158, 0.12954092684266635]
[(63, 0.27910418255245845)]


The actual generation of sequences

In [ ]:
import pandas as pd

sur_factors = [
    (True, True),
    (True, False),
    (False, True),
    (False, False)
]

pos_factors = [
    (2,3),
    (4,5),
    (6,7)
]

change_factors = [
    True,
    False
]

gen_factors = [
    True,
    False
]

probe_tones = [
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74
]

length = 8

data = {
    "Generated": [],
    "Change": [],
    "Position": [],
    "Surprisal": [],
    "Probe": [],
    "Sequence": [],
    "Probabilites": [],
    "Alternatives": []
}

for gen in gen_factors:
    for change in change_factors:
        for pos in pos_factors:
            for surprise in sur_factors:
                probe = random.choice(probe_tones)
                if change:
                    position = random.choice(pos)
                    if gen:
                        mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                        while mel == False:
                            mel = GenerateBinaryTreeWithChange(probe, length, position, surprise)
                        seq, probs, alt = mel
                    else:
                        mel = GenerateMelodyWithChange(probe, length, position, surprise)
                        while mel == False:
                            mel = GenerateMelodyWithChange(probe, length, position, surprise)
                        seq, probs, alt = mel
                else:
                    position = -1
                    surprise = False
                    if gen:
                        seq, probs = GenerateBinaryTree(probe, length)
                        alt = False
                    else:
                        mel = GenerateMelody(probe, length)
                        seq, probs = mel
                        alt = False

                data["Generated"].append(gen)
                data["Change"].append(change)
                data["Position"].append(position)
                data["Surprisal"].append(surprise)
                data["Probe"].append(probe)
                data["Sequence"].append(seq)
                data["Probabilites"].append(probs)
                data["Alternatives"].append(alt)

df = pd.DataFrame(data)
df.to_csv("sequences.csv", index=False)

In [140]:

seq = [
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74
]

print(ConvertToFreqs(seq))

[261.626, 277.183, 293.665, 311.127, 329.628, 349.228, 369.994, 391.995, 415.305, 440.0, 466.164, 493.883, 523.251, 554.365, 587.33]
